> 🔑 **SOLUTIONS NOTEBOOK**: instructor / self-check copy. Try the problems in `../notebooks/` first!

# 🏆 Lesson 09: Practical Assignment: **The Regression Showdown**

**Time:** 3–4 hours  **Prerequisites:** Lessons 00–08  **This is graded work** (self-graded with the rubric below, or by an instructor).

---

## 📋 The assignment

> **Train several regression models using the same dataset. Compare:** Linear Regression · Decision Tree · Random Forest · Gradient Boosting · XGBoost.
> **Evaluate their performance and explain why one model performs better than another.**

### 🎯 Expected outcome
You should be able to **optimise models, compare competing approaches and make evidence-based model selections.**

### 🧭 The story
You are the data scientist at a property platform in the city of Eastbridge. Product wants a **price-estimate feature** for new listings (the `housing.csv` data). Your manager says:

*"I don't care which algorithm is fashionable. Pick the model you can defend with evidence, tell me **why** it wins, and tell me what drives its predictions."*

### 📦 Deliverables
1. **This notebook**, run top to bottom with every milestone completed.
2. A **written analysis** (≈ 250–400 words) in the last markdown cell: the winner, the evidence, *why* it wins, caveats.

### ⚖️ Rubric (100 points)

| criterion | points | what graders look for |
|---|---|---|
| **Correct pipelines** | 15 | all five models, preprocessing inside `Pipeline`s, no leakage, fixed seeds |
| **Sound evaluation protocol** | 20 | repeated CV on the *training* set for model choice; test set touched **once**; paired folds |
| **Analysis of results** | 20 | tables + plots; segment/residual diagnostics; effect of log-target; a **significance** statement |
| **Explanation of *why*** | 25 | links model behaviour to **data properties** (non-linearity, interactions, skew, outliers, variance) with evidence from *your* results |
| **Interpretation** | 10 | feature importance / SHAP read correctly; no causal over-claims |
| **Communication** | 10 | clear, honest, concise; limitations stated |
| **Bonus** | +5 | any stretch challenge (stacking, prediction intervals, tuning) done properly |

### 📏 Ground rules
* The **test set is sealed** until Milestone 4. Model *choice* uses cross-validation on the training set.
* Use the **model settings given in Milestone 2** so everyone's numbers are comparable (you may explore others *after* the baseline comparison).
* Run cells **in order**: later cells depend on your earlier answers. Self-checks (`✅`) tell you whether a milestone is on track.

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
import xgboost as xgb
import seaborn as sns

## Milestone 1: Know your data (and your floor)

Before modelling, look at the target and the baseline. Run the cells, then answer the first problem.

In [2]:
X_train, X_test, y_train, y_test = get_housing_split()
print(f"train {X_train.shape}   test {X_test.shape}   (test set is SEALED until Milestone 4)")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist(y_train, bins=50, color=PALETTE["blue"]); axes[0].set_title(f"Target price_k  (skew = {y_train.skew():.1f})")
axes[1].hist(np.log(y_train), bins=50, color=PALETTE["orange"]); axes[1].set_title("log(price_k)")
axes[2].scatter(X_train["size_sqm"], y_train, s=5, alpha=.35); axes[2].set_title("Size vs price: fan-shaped, not linear"); axes[2].set_xlabel("size_sqm")
plt.tight_layout(); plt.show()
print("missing values in the training features:"); print(X_train.isna().sum()[X_train.isna().any()].to_string())

train (2400, 17)   test (600, 17)   (test set is SEALED until Milestone 4)
missing values in the training features:
age_years          100
school_rating      142
security_rating     60
energy_rating      212


### 🧩 Problem 09.1 ⭐: The floor

Compute the **test RMSE of the dumbest sensible model**: always predicting the *training* mean price, and store it in `baseline_rmse`.
Every model you build must beat this number comfortably.

<details><summary>💡 Hint 1</summary>

The baseline prediction is a constant: `y_train.mean()`.

</details>

<details><summary>💡 Hint 2</summary>

RMSE = `np.sqrt(np.mean((y_test - constant) ** 2))`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [3]:
baseline_rmse = float(np.sqrt(np.mean((y_test - y_train.mean()) ** 2)))

# ---- self-check ----
check("09.1", lambda: 300 < baseline_rmse < 420, "Expect roughly 350 (≈ the standard deviation of prices).", needs=[baseline_rmse])

✅ 09.1: correct -- nice work!


**✅ Solution walk-through (Problem 09.1)**

≈ $356k: a model that ignores the features entirely is wrong by about the standard deviation of prices.

## Milestone 2: Build the five pipelines

Use **exactly** these settings (so results are comparable across the class):

| name (dict key) | estimator |
|---|---|
| `"Linear Regression"` | `LinearRegression()` |
| `"Decision Tree"` | `DecisionTreeRegressor(random_state=SEED)` *(no depth limit; the textbook unpruned tree)* |
| `"Random Forest"` | `RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)` |
| `"Gradient Boosting"` | `GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=SEED)` |
| `"XGBoost"` | `xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, random_state=SEED)` |

Each must be a `Pipeline`: **`make_preprocessor(X_train, scale=True)`** (median-impute, scale, one-hot) followed by the estimator.
Why the preprocessor inside the pipeline? (Lesson 06: so CV re-fits it on every training fold; no leakage.)

### 🧩 Problem 09.2 ⭐⭐: Assemble the models

Write `build_models(seed=SEED)` returning a dict `{name: Pipeline}` with the five entries above, **in that order**.
Then call it: `models = build_models()`.

<details><summary>💡 Hint 1</summary>

`make_pipeline(make_preprocessor(X_train, scale=True), LinearRegression())` is the first one.

</details>

<details><summary>💡 Hint 2</summary>

Create a *fresh* preprocessor for each pipeline (call `make_preprocessor` inside each `make_pipeline`).

</details>

<details><summary>💡 Hint 3</summary>

Pipelines are lists of steps: `pipe[-1]` is the final estimator: handy for checking.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [4]:
def build_models(seed=SEED):
    prep = lambda: make_preprocessor(X_train, scale=True)
    return {
        "Linear Regression": make_pipeline(prep(), LinearRegression()),
        "Decision Tree": make_pipeline(prep(), DecisionTreeRegressor(random_state=seed)),
        "Random Forest": make_pipeline(prep(), RandomForestRegressor(n_estimators=300, random_state=seed, n_jobs=-1)),
        "Gradient Boosting": make_pipeline(prep(), GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                                                                             subsample=0.8, random_state=seed)),
        "XGBoost": make_pipeline(prep(), xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8,
                                                          colsample_bytree=0.8, random_state=seed)),
    }

models = build_models()

# ---- self-check ----
def _ok():
    want = ["Linear Regression", "Decision Tree", "Random Forest", "Gradient Boosting", "XGBoost"]
    kinds = ["LinearRegression", "DecisionTreeRegressor", "RandomForestRegressor", "GradientBoostingRegressor", "XGBRegressor"]
    return (list(models) == want and all(type(models[n][-1]).__name__ == k for n, k in zip(want, kinds))
            and all(type(models[n]).__name__ == "Pipeline" for n in want))
check("09.2", _ok, "Five Pipelines, keyed by the exact names, each ending in the right estimator class.", needs=[models])

✅ 09.2: correct -- nice work!


**✅ Solution walk-through (Problem 09.2)**

A dict of Pipelines makes every later step a loop: `for name, pipe in models.items()`: which is how you keep comparisons fair: the same code path for every model.

## Milestone 3: Choose with cross-validation (the test set stays sealed!)

Evaluate each pipeline with **repeated 5-fold CV (2 repeats = 10 folds) on the training set**, using the *same folds for every model* (a fixed `RepeatedKFold` seed ⇒ paired comparison, Lesson 06).

### 🧩 Problem 09.3 ⭐⭐: Repeated cross-validation

Use `cross_validate` with `rkf` (provided) on `(X_train, y_train)` for every model, scoring `rmse` (`neg_root_mean_squared_error`), `mae` (`neg_mean_absolute_error`) and `r2`. Build:

* `fold_rmse`: a DataFrame, **10 rows (folds) × 5 columns (models)** of per-fold RMSE (positive numbers!)
* `cv_summary`: a DataFrame indexed by model name with columns `RMSE_mean`, `RMSE_std`, `MAE_mean`, `R2_mean`, `fit_seconds`

<details><summary>💡 Hint 1</summary>

`res = cross_validate(pipe, X_train, y_train, cv=rkf, scoring={'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'})`.

</details>

<details><summary>💡 Hint 2</summary>

`res['test_rmse']` is negative (scikit-learn maximises scores); flip the sign. `res['fit_time']` holds seconds per fold.

</details>

<details><summary>💡 Hint 3</summary>

Collect `fold_rmse[name] = -res['test_rmse']` in a dict, then `pd.DataFrame(fold_rmse)`; use `std(ddof=1)` for `RMSE_std`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [5]:
rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=SEED)

fold_dict, rows = {}, []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=rkf, n_jobs=1,
                         scoring={"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"})
    fold_dict[name] = -res["test_rmse"]
    rows.append({"model": name, "RMSE_mean": (-res["test_rmse"]).mean(), "RMSE_std": (-res["test_rmse"]).std(ddof=1),
                 "MAE_mean": (-res["test_mae"]).mean(), "R2_mean": res["test_r2"].mean(), "fit_seconds": res["fit_time"].mean()})
fold_rmse = pd.DataFrame(fold_dict)
cv_summary = pd.DataFrame(rows).set_index("model")

# ---- self-check ----
def _ok():
    cols = {"RMSE_mean", "RMSE_std", "MAE_mean", "R2_mean", "fit_seconds"}
    return (fold_rmse.shape == (10, 5) and (fold_rmse.values > 0).all() and cols <= set(cv_summary.columns) and cv_summary.shape[0] == 5
            and cv_summary["RMSE_mean"].idxmin() in ("Gradient Boosting", "XGBoost", "Random Forest")
            and cv_summary.loc["Decision Tree", "RMSE_mean"] > cv_summary.loc["Random Forest", "RMSE_mean"])
check("09.3", _ok, "fold_rmse must be 10×5 with positive RMSEs; the best CV model should be an ensemble, and the single tree must lose to the forest.",
      needs=[fold_rmse, cv_summary])

✅ 09.3: correct -- nice work!


**✅ Solution walk-through (Problem 09.3)**

Every model sees identical folds, so differences are *paired*. `cross_validate` clones the pipeline for each fold, so the `models` dict stays unfitted and the preprocessing is re-fitted on each training fold only.

In [6]:
cv_summary.sort_values("RMSE_mean").round(2)

,RMSE_mean,RMSE_std,MAE_mean,R2_mean,fit_seconds
model,,,,,
XGBoost,114.94,18.99,60.34,0.90,0.11
Gradient Boosting,119.17,20.27,64.22,0.89,0.60
Random Forest,142.05,22.78,79.88,0.85,2.10
Linear Regression,162.84,19.43,103.29,0.80,0.02
Decision Tree,219.03,30.92,123.95,0.62,0.02


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = cv_summary.sort_values("RMSE_mean").index
sns.boxplot(data=fold_rmse[list(order)], ax=axes[0], color=PALETTE["blue"]); axes[0].set_title("10 fold-RMSEs per model"); axes[0].tick_params(axis="x", rotation=20)
for i in range(len(fold_rmse)):
    axes[1].plot(range(5), fold_rmse.loc[i, order], color=PALETTE["gray"], alpha=.4)
axes[1].plot(range(5), fold_rmse[order].mean(), "o-", color=PALETTE["red"], lw=2.5); axes[1].set_xticks(range(5)); axes[1].set_xticklabels(order, rotation=20)
axes[1].set_title("Same folds for every model (paired)")
plt.tight_layout(); plt.show()

## Milestone 4: The single, final test-set evaluation

Now (and only now) refit every pipeline on the **full training set** and score it on the **sealed test set**. This is the number you will quote to your manager. *Touch the test set once.*

### 🧩 Problem 09.4 ⭐: Refit and test

* `fitted`: fit each pipeline in `models` on `(X_train, y_train)` and keep them in a dict (same keys)
* `test_preds`: dict of each model's **test-set predictions** (NumPy arrays)
* `test_table`: DataFrame indexed by model with columns `RMSE, MAE, R2, MAPE_%` (use `regression_report`), sorted by `RMSE` ascending

<details><summary>💡 Hint 1</summary>

`fitted = {name: pipe.fit(X_train, y_train) for name, pipe in models.items()}` (`fit` returns the pipeline).

</details>

<details><summary>💡 Hint 2</summary>

`test_preds = {name: pipe.predict(X_test) for name, pipe in fitted.items()}`.

</details>

<details><summary>💡 Hint 3</summary>

`pd.DataFrame({n: regression_report(y_test, p) for n, p in test_preds.items()}).T.sort_values('RMSE')`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [8]:
fitted = {name: pipe.fit(X_train, y_train) for name, pipe in models.items()}
test_preds = {name: pipe.predict(X_test) for name, pipe in fitted.items()}
test_table = pd.DataFrame({n: regression_report(y_test, p) for n, p in test_preds.items()}).T.sort_values("RMSE")

# ---- self-check ----
check("09.4", lambda: list(fitted) == list(models) and set(test_preds) == set(models) and test_table.shape[0] == 5
      and {"RMSE", "MAE", "R2"} <= set(test_table.columns) and test_table["RMSE"].is_monotonic_increasing
      and test_table["RMSE"].min() < 0.5 * baseline_rmse and len(test_preds["XGBoost"]) == len(y_test),
      "Five fitted pipelines, predictions for 600 test rows, and a table sorted by RMSE; the best model should beat the baseline by > 50%.",
      needs=[fitted, test_preds, test_table])

✅ 09.4: correct -- nice work!


**✅ Solution walk-through (Problem 09.4)**

Same loop, same data, one call per model. The best ensemble cuts the error to less than a third of the baseline.

In [9]:
best_name = test_table.index[0]
print(f"🏆 lowest test RMSE: {best_name}")
test_table.round(3)

🏆 lowest test RMSE: XGBoost


,RMSE,MAE,R2,MAPE_%
XGBoost,94.730,55.701,0.929,12.527
Gradient Boosting,97.378,60.105,0.925,14.009
Random Forest,122.980,78.722,0.881,18.832
Linear Regression,149.916,100.460,0.823,33.816
Decision Tree,214.839,122.139,0.637,26.678


In [10]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
lim = [0, float(np.percentile(y_test, 99.5))]
for ax, name in zip(axes.ravel(), test_table.index):
    p = test_preds[name]
    ax.scatter(y_test, p, s=7, alpha=.4, color=PALETTE["blue"]); ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_title(f"{name}   RMSE {test_table.loc[name, 'RMSE']:.0f}")
    ax.set_xlabel("actual price_k"); ax.set_ylabel("predicted")
axes.ravel()[-1].axis("off")
plt.suptitle("Predicted vs actual (test set): closer to the dashed line is better", y=1.0); plt.tight_layout(); plt.show()

In [11]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.4), sharey=True)
for ax, name in zip(axes, test_table.index):
    resid = y_test.values - test_preds[name]
    ax.scatter(test_preds[name], resid, s=6, alpha=.35, color=PALETTE["purple"]); ax.axhline(0, color="k", lw=1)
    ax.set_title(name, fontsize=10); ax.set_xlabel("predicted")
axes[0].set_ylabel("residual (actual − predicted)")
plt.suptitle("Residuals vs predictions; look for fans, curves and outliers", y=1.04); plt.tight_layout(); plt.show()

**Questions to ask of these plots** (you'll need the answers for your written analysis):
* Which models' residuals **fan out** as the prediction grows? Which show a **curve** (systematic bias)?
* Which model has the tightest cloud around the dashed line? Are there **points no model gets right**?
* Does the *linear* model do something qualitatively different from the tree models?

## Milestone 5: Diagnose: *where* do the models fail?

A single RMSE hides structure. Split the **test set into three price terciles** (low / mid / high) and compute each model's RMSE inside each segment.

### 🧩 Problem 09.5 ⭐⭐: Error by price segment

Build `seg_table`: a DataFrame with index `["low", "mid", "high"]` and one column per model, containing the **RMSE within each price tercile of the test set** (terciles of the *true* test price, `pd.qcut(y_test, 3, labels=[...])`).

<details><summary>💡 Hint 1</summary>

`segment = pd.qcut(y_test, 3, labels=['low', 'mid', 'high'])`.

</details>

<details><summary>💡 Hint 2</summary>

For each segment `s` and model `n`: `mask = (segment == s).values`; `rmse = np.sqrt(np.mean((y_test.values[mask] - test_preds[n][mask]) ** 2))`.

</details>

<details><summary>💡 Hint 3</summary>

Use a nested dict `{model: [rmse_low, rmse_mid, rmse_high]}` and `pd.DataFrame(d, index=['low', 'mid', 'high'])`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [12]:
segment = pd.qcut(y_test, 3, labels=["low", "mid", "high"])
seg_table = pd.DataFrame(
    {n: [float(np.sqrt(np.mean((y_test.values[(segment == s).values] - test_preds[n][(segment == s).values]) ** 2)))
         for s in ["low", "mid", "high"]] for n in test_preds},
    index=["low", "mid", "high"])

# ---- self-check ----
check("09.5", lambda: seg_table.shape == (3, 5) and list(seg_table.index) == ["low", "mid", "high"]
      and (seg_table.loc["high"] > seg_table.loc["low"]).all(),
      "3 rows × 5 columns; absolute errors should grow with the price level for every model.", needs=[seg_table])

✅ 09.5: correct -- nice work!


**✅ Solution walk-through (Problem 09.5)**

Absolute errors grow with price for *every* model (a $1M house has a bigger dollar error than a $200k flat at the same percentage accuracy): the fan-shaped heteroscedasticity visible in the residual plots. This is a strong hint that the **target should be modelled on a log scale**: Milestone 6.

In [13]:
print("RMSE ($k) within each price tercile of the test set")
seg_table.round(0).astype(int)

RMSE ($k) within each price tercile of the test set


,Linear Regression,Decision Tree,Random Forest,Gradient Boosting,XGBoost
low,116,59,50,35,35
mid,89,129,70,54,52
high,215,344,195,156,152


## Milestone 6: Improve with insight: the log-target experiment

Prices are multiplicative (a pool adds ~12%, not $40k) and right-skewed. Models that fit **`log(price)`** and exponentiate back (`TransformedTargetRegressor`) work with *percentage* effects, which is often a much better inductive bias, especially for **linear** models. Test the idea on the weakest *and* the strongest model family.

### 🧩 Problem 09.6 ⭐⭐: Does a log target help?

Using `TransformedTargetRegressor(regressor=<pipeline>, func=np.log, inverse_func=np.exp)`:

* wrap a *fresh* linear pipeline (from `build_models()`) → fit on train → `rmse_linear_log` (test RMSE)
* wrap a *fresh* XGBoost pipeline likewise → `rmse_xgb_log`
* and read the *raw-target* numbers from `test_table`: `rmse_linear_raw`, `rmse_xgb_raw`

<details><summary>💡 Hint 1</summary>

`fresh = build_models()`; `lin_log = TransformedTargetRegressor(regressor=fresh['Linear Regression'], func=np.log, inverse_func=np.exp).fit(X_train, y_train)`.

</details>

<details><summary>💡 Hint 2</summary>

`rmse_linear_log = regression_report(y_test, lin_log.predict(X_test))['RMSE']`.

</details>

<details><summary>💡 Hint 3</summary>

Raw numbers: `test_table.loc['Linear Regression', 'RMSE']` and `test_table.loc['XGBoost', 'RMSE']`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [14]:
rmse_linear_raw = float(test_table.loc["Linear Regression", "RMSE"])
rmse_xgb_raw = float(test_table.loc["XGBoost", "RMSE"])

fresh = build_models()
def log_fit(name):
    m = TransformedTargetRegressor(regressor=fresh[name], func=np.log, inverse_func=np.exp).fit(X_train, y_train)
    return regression_report(y_test, m.predict(X_test))["RMSE"]
rmse_linear_log = log_fit("Linear Regression")
rmse_xgb_log = log_fit("XGBoost")

# ---- self-check ----
check("09.6", lambda: rmse_linear_log < 0.85 * rmse_linear_raw and rmse_xgb_log < rmse_xgb_raw,
      "A log target should cut the linear model's error dramatically (> 15%) and also help XGBoost.",
      needs=[rmse_linear_raw, rmse_xgb_raw, rmse_linear_log, rmse_xgb_log])

✅ 09.6: correct -- nice work!


**✅ Solution walk-through (Problem 09.6)**

The linear model improves enormously (≈ 150 → ≈ 108): once price effects are additive in log space, a *linear* model suddenly fits a multiplicative world. XGBoost improves too (≈ 95 → ≈ 84): the tree ensemble learns piece-wise constants, and on the log scale equal *relative* errors are treated equally, so it stops obsessing over the dollar errors of the priciest houses. **Insight beats compute.**

## Milestone 7: Is the winner *really* better? (statistics + selection)

Take the **two models with the lowest CV mean RMSE** (from `cv_summary`, not from the test set!). Using the 10 paired fold scores, test whether the difference is real with the **corrected resampled t-test** (`corrected_resampled_ttest(diffs, n_train, n_test)`: the one you wrote in Lesson 06; each CV fold trains on 1,920 rows and validates on 480). Then apply the **one-standard-error rule**.

### 🧩 Problem 09.7 ⭐⭐⭐: Evidence-based selection

* `top2`: list of the two model names with the lowest `RMSE_mean`
* `p_value`: corrected p-value for the paired fold differences between them
* `significant`: `True` if `p_value < 0.05`
* `chosen_model`: apply the **one-SE rule**: among models whose `RMSE_mean` is within **one standard error** (`RMSE_std/sqrt(10)` of the *best* model) of the best, choose the **simplest**, using this complexity order: `Linear Regression=1, Decision Tree=2, Gradient Boosting=3, XGBoost=4, Random Forest=5`.

<details><summary>💡 Hint 1</summary>

`top2 = cv_summary['RMSE_mean'].nsmallest(2).index.tolist()`.

</details>

<details><summary>💡 Hint 2</summary>

`d = (fold_rmse[top2[0]] - fold_rmse[top2[1]]).values`; `t, p = corrected_resampled_ttest(d, n_train=1920, n_test=480)`.

</details>

<details><summary>💡 Hint 3</summary>

`se = cv_summary.loc[best, 'RMSE_std'] / np.sqrt(10)`; eligible = models with `RMSE_mean <= best_mean + se`; `chosen_model = min(eligible, key=complexity.get)`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [15]:
complexity = {"Linear Regression": 1, "Decision Tree": 2, "Gradient Boosting": 3, "XGBoost": 4, "Random Forest": 5}

top2 = cv_summary["RMSE_mean"].nsmallest(2).index.tolist()
d = (fold_rmse[top2[0]] - fold_rmse[top2[1]]).values
t_stat, p_value = corrected_resampled_ttest(d, n_train=1920, n_test=480)
significant = bool(p_value < 0.05)

best = top2[0]
se = cv_summary.loc[best, "RMSE_std"] / np.sqrt(10)
eligible = [m for m in cv_summary.index if cv_summary.loc[m, "RMSE_mean"] <= cv_summary.loc[best, "RMSE_mean"] + se]
chosen_model = min(eligible, key=complexity.get)

# ---- self-check ----
def _ref():
    cx = {"Linear Regression": 1, "Decision Tree": 2, "Gradient Boosting": 3, "XGBoost": 4, "Random Forest": 5}
    t2 = cv_summary["RMSE_mean"].nsmallest(2).index.tolist()
    dd = (fold_rmse[t2[0]] - fold_rmse[t2[1]]).values
    _, p = corrected_resampled_ttest(dd, 1920, 480)
    b = t2[0]; se_ = cv_summary.loc[b, "RMSE_std"] / np.sqrt(10)
    el = [m for m in cv_summary.index if cv_summary.loc[m, "RMSE_mean"] <= cv_summary.loc[b, "RMSE_mean"] + se_]
    return t2, p, min(el, key=cx.get)
check("09.7", lambda: top2 == _ref()[0] and abs(p_value - _ref()[1]) < 1e-9 and significant == bool(_ref()[1] < 0.05) and chosen_model == _ref()[2],
      "Use CV means (not test scores), the corrected test with n_train=1920 / n_test=480, and the one-SE rule with the given complexity order.",
      needs=[top2, p_value, significant, chosen_model])

✅ 09.7: correct -- nice work!


**✅ Solution walk-through (Problem 09.7)**

Selecting on **CV** (not test) keeps the test set honest. In our run the two best models are XGBoost and Gradient Boosting; XGBoost wins 6 of 10 folds by ≈ 4 RMSE points, but the corrected test gives p ≈ 0.31: **not significant**: so the one-SE rule selects the *simpler* Gradient Boosting model. (Note the test-set winner is XGBoost: that is fine. We do not switch models because of a test-set gap; the test set only *confirms* the choice.) This is what *evidence-based model selection* looks like.

## Milestone 8: What drives the winner's predictions?

Use **permutation importance on the test set** (Lesson 08) for the pipeline you **chose** in Milestone 7 (`chosen_model`). Because the pipeline includes the preprocessing, we can permute the **raw** columns directly.

### 🧩 Problem 09.8 ⭐⭐: Top drivers

Compute permutation importance (`n_repeats=5, random_state=SEED, scoring="neg_root_mean_squared_error"`) for `fitted[chosen_model]` on `(X_test, y_test)`.
Store the **five most important raw feature names** (largest mean importance first) in `top5`.

<details><summary>💡 Hint 1</summary>

`pi = permutation_importance(fitted[chosen_model], X_test, y_test, n_repeats=5, random_state=SEED, scoring='neg_root_mean_squared_error')`.

</details>

<details><summary>💡 Hint 2</summary>

`pi.importances_mean` aligns with `X_test.columns`: `pd.Series(pi.importances_mean, index=X_test.columns).nlargest(5).index.tolist()`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [16]:
pi = permutation_importance(fitted[chosen_model], X_test, y_test, n_repeats=5, random_state=SEED, scoring="neg_root_mean_squared_error", n_jobs=1)
top5 = pd.Series(pi.importances_mean, index=X_test.columns).nlargest(5).index.tolist()

# ---- self-check ----
check("09.8", lambda: len(top5) == 5 and "size_sqm" in top5 and set(top5) <= set(X_test.columns),
      "Five raw column names; size_sqm should certainly be among them.", needs=[top5])

✅ 09.8: correct -- nice work!


**✅ Solution walk-through (Problem 09.8)**

Size, neighbourhood and distance-to-centre dominate: matching the hidden recipe. Reporting the drivers is part of the deliverable: a model nobody can explain is hard to ship.

## Milestone 9: Your written analysis (the part that earns 25 + 10 points)

Write **250–400 words** in the markdown cell below. Use *your own numbers*. Suggested structure:

1. **Verdict (2 sentences).** Which model do you recommend and by how much does it beat the alternatives (CV mean ± spread; test RMSE)?
2. **Evidence (3–4 sentences).** Cite the CV table, the paired-fold test / one-SE rule, and the segment table. Is the win statistically meaningful *and* practically meaningful?
3. **Why it wins: link each model to a data property (the heart of the assignment).** Answer with evidence:
   * Why is **Linear Regression** weak on raw prices? *(think: multiplicative effects, thresholds, interactions, skew; what did the log-target experiment show?)*
   * Why does the **single Decision Tree** underperform, and why is it *worse* than the linear model here? *(variance! compare with the forest)*
   * Why does the **Random Forest** fix the tree's problem, and what limits it? *(correlated trees, no extrapolation, hard leaf averages in the tails)*
   * Why do the **boosted models** do best? *(sequential bias reduction, shallow trees → interactions, shrinkage)* How do **Gradient Boosting vs XGBoost** differ in practice here?
4. **What drives predictions (1–2 sentences)** from your importance analysis, with the *correlation ≠ causation* caveat.
5. **Caveats & next steps (2–3 sentences):** where does the model fail (high-price tail?), what would you try next (tuning, log target on all models, stacking, more data), and what monitoring would you set up (drift, time-forward validation; Lesson 06)?

> ✍️ **Your analysis:** *(double-click this cell and write here)*

### 🔑 Model answer (for self-checking: written from the numbers produced by this very notebook)

**Verdict.** I recommend a **gradient-boosted tree model**. By cross-validation (10 paired folds, training set only) XGBoost (RMSE 114.9 ± 19.0) and sklearn's Gradient Boosting (119.2 ± 20.3) clearly beat the Random Forest (142.1), Linear Regression (162.8) and the single Decision Tree (219.0). On the sealed test set the ranking is the same: 94.7 / 97.4 / 123.0 / 149.9 / 214.8, against a baseline of 356.

**Evidence.** The boosted models lead by 20+ RMSE points in the CV table, and in every price segment (e.g. high tercile: XGBoost 152, GB 156, RF 195, linear 215, tree 344). But *between* XGBoost and Gradient Boosting the difference (≈ 4 RMSE, XGBoost better in 6 of 10 folds) is **not significant** under the corrected resampled t-test (p ≈ 0.31). The one-standard-error rule therefore selects the *simpler* Gradient Boosting model. XGBoost's slightly lower test RMSE is exactly the kind of gap a single split produces by luck, so I would not switch based on it, though XGBoost trains ~4× faster (0.15 s vs 0.6 s per fit), which is a legitimate tie-breaker for retraining.

**Why each model lands where it does.**
* *Linear Regression* assumes additive, straight-line effects, but this market is **multiplicative and thresholded**: size matters more in prime neighbourhoods (interaction), transit time has a step near 8 minutes, school quality only helps above ~6. The residuals fan out and its MAPE is 34%. A log target cuts its test RMSE from 149.9 to 108.1 (−28%): proof that much of its weakness was the *multiplicative structure*, not just non-linearity.
* *The single Decision Tree* captures the non-linearity but is **high-variance**: unpruned, it memorises noise (and the ×2 outlier "mansions"), so it is even worse than the linear model on RMSE (214.8) with a terrible high-price error (344).
* *The Random Forest* averages many such trees, removing most of that variance (214.8 → 123.0, −43%). Two limits: its trees are correlated (with scikit-learn's default `max_features=1.0` it is effectively bagging; Lesson 02 showed lower values reduce the correlation), and it predicts leaf averages, so it is weakest in the sparse expensive tail.
* *Boosting* wins because it **reduces bias** sequentially: shallow trees (depth 3–4) capture the interactions and thresholds, shrinkage and subsampling control variance, and each tree corrects the previous errors: here a further ~20% below the forest.

**Drivers.** Permutation importance puts `size_sqm`, `distance_to_center_km`, `neighborhood`, `school_rating` and `age_years` on top. These are the model's drivers, not proven causes.

**Caveats / next steps.** Absolute errors remain large for expensive houses (≈ 150 in the top tercile). A **log target** also helps XGBoost (94.7 → 84.3), so I would adopt it, then tune (Optuna) and validate **forward in time** because prices inflate ≈ 5%/year and trees cannot extrapolate. I would monitor drift and retrain regularly.

## 🚀 Stretch challenges (+5 bonus points for any one done properly)

1. **Stack them.** Build a stacking ensemble (non-negative linear meta-learner, OOF predictions, Lesson 05) from your five models *plus* a log-target linear model. Does it beat the best single model on the test set? Is the gain statistically meaningful on CV?
2. **Tune the winner.** Run an Optuna study (Lesson 07) with ≥ 40 trials on XGBoost *with a log target*, using CV on the training set only. Report the CV gain and the single test-set number.
3. **Prediction intervals.** Use LightGBM's `objective="quantile"` (alpha 0.1 and 0.9) to produce 80% intervals. What fraction of test prices fall inside? (Aim for ≈ 80%.) Where are the intervals widest, and why?
4. **Time-forward check.** Re-evaluate the five models with forward-chaining by `year_listed` (Lesson 06). Does the ranking change? What does that say about deploying a forest in a market with inflation?
5. **Monotonic constraints.** Constrain XGBoost so price is non-decreasing in `size_sqm`. Does accuracy suffer? Why might a business want this?

## ✅ Self-assessment checklist
- [ ] Preprocessing lives **inside** the pipelines (no leakage)
- [ ] Model choice used **CV on the training set**; the test set was evaluated **once**
- [ ] I reported **mean ± spread**, not a single number, and a **paired** comparison
- [ ] I looked at **segments and residuals**, not only RMSE
- [ ] I tried at least one **data-aware improvement** (log target) and measured it
- [ ] My explanation connects each model to a **property of the data**, with evidence
- [ ] I stated **limitations** and avoided causal claims

In [17]:
scoreboard()

Scoreboard: 8 solved, 0 to fix, 0 not attempted
   ✅ 09.1
   ✅ 09.2
   ✅ 09.3
   ✅ 09.4
   ✅ 09.5
   ✅ 09.6
   ✅ 09.7
   ✅ 09.8
